# The portal package, one small step at a time

Same gentle idea as the CMS notebook. No agent, nothing clever. One cell, one
small thing. Press **Shift + Enter** and read the result.

This package reads your **grades** and your **transcript** from the student
portal.

Note: the portal is old and slow. A cell may take up to a minute to answer. That
is normal here. Just wait for it.

## Step 1 — log in

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # reads your login from .env if it is there
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

from guc_portal import GucPortal

portal = GucPortal()
print("Logged in. ")

### Using GIU instead of GUC

This talks to **GUC** by default. To use **GIU** instead, pick one:

- add a line `GUC_SITE=giu` to your `.env` file, then run the cell above again, or
- change the cell above to `portal = GucPortal(site="giu")`.

Your GIU username and password go in the same `GUC_USERNAME` / `GUC_PASSWORD`.
Everything below stays exactly the same. (GIU is newer here, so a page or two may
still need small fixes.)

## Step 2 — your terms

`available_seasons()` lists the terms you have grades for.

In [ ]:
import re
import ipywidgets as widgets
from IPython.display import display

# GUC's semesters in the order they actually happen inside a calendar year.
# "Winter" is the autumn term (starts ~October), so it comes LAST in its year.
_SEASON_RANK = {"spring": 0, "summer": 1, "fall": 2, "winter": 3}

def _term_key(name):
    match = re.search(r"(\w+)\s+(\d{4})", name)
    if not match:
        return (0, 0)
    season, year = match.groups()
    return (int(year), _SEASON_RANK.get(season.lower(), -1))

terms = sorted(portal.available_seasons(), key=lambda t: _term_key(t[1]))

if not terms:
    print("No previous terms found. This page only lists *past* semesters, so if")
    print("this is your first term it will be empty - try Step 5/6 (transcript)")
    print("or Step 4 (current-term grades) instead.")
else:
    print("Your terms:\n")
    for code, name in terms:
        print(" -", name)

    term_dropdown = widgets.Dropdown(
        options=[(name, code) for code, name in terms],
        description="Term:",
    )
    display(term_dropdown)

## Step 3 — the courses in one term

`list_previous_courses(term_code)` lists the courses you took in one term. It is
handy for finding a course's exact name before you ask for its grades.

Pick a term from the dropdown in Step 2, then run this cell.

In [ ]:
code = term_dropdown.value
name = term_dropdown.label
if code:
    print("Courses in", name, ":\n")
    for c_code, c_name in portal.list_previous_courses(code):
        print(" -", c_name)
else:
    print("No term selected - pick one from the Step 2 dropdown.")

## Step 4 — your grades in one course

`get_grades_by_name(term, course)` gives every quiz and assignment mark for one
course, written as earned / max.

In [ ]:
grades = portal.get_grades_by_name("Winter 2024", "Discrete Math")
print(grades.course, "(", grades.season, ")\n")
for item in grades.items:
    print(" ", item.assessment, ":", item.grade)

## Step 5 — your study years

`available_years()` lists the years on your transcript. Each has a short code.

The transcript is the busiest part of the portal, so if it is not ready we show
a gentle note instead of a scary error. If you see the note, just wait a minute
and run the cell again.

In [ ]:
try:
    for code, name in portal.available_years():
        print(code, "=", name)
except Exception:
    print("The portal is busy right now. Wait a minute, then run this cell again.")

## Step 6 — your transcript for one year

`get_transcript_year(code)` gives that year's courses and your GPA. Use a code from Step 5.

In [ ]:
try:
    t = portal.get_transcript_year("22", tries=1)   # "22" is the code for 2024-2025 (see above)
    print("Cumulative GPA:", t.cumulative_gpa, "\n")
    for row in t.rows:
        print(" ", row.grade, "-", row.course)
except Exception:
    print("The portal is busy right now. Wait a minute, then run this cell again.")

## That is all

You just used the whole portal package:

- `available_seasons()` — your terms
- `list_previous_courses(term_code)` — the courses in a term
- `get_grades_by_name(term, course)` — your marks in a course
- `available_years()` — your study years
- `get_transcript_year(code)` — a year's courses and GPA

No agent, no magic. Just a few functions that read the portal for you. If a cell
gives an error, the portal was busy: wait a minute and run it again.